# explore_sql -- Ad-hoc SQL against local Delta table cache

**Flow**
1. Sync cell -- downloads Parquet files from ADLS to `.cache/` (run once, then when you need fresh data)
2. Register cell -- mounts the local cache into a DuckDB connection
3. Query cells -- `conn.sql()` against the registered tables

Re-run the Sync cell at any time. Only files added since last sync are downloaded -- no full re-download.

## Imports

In [ ]:
import os
import duckdb
from pathlib import Path
from lakekit.sync import sync_table, register_local, CACHE_DIR

## Sync -- download / incrementally update local Parquet cache

Each entry: `(alias, container, path_in_container, adls_account)`

Add or remove rows to control which tables are cached locally.

In [ ]:
ADLS_ACCOUNT = os.environ["ADLS_ACCOUNT"]

TABLES = [
    ("raw_events",     "bronze", "events/raw_events/",     ADLS_ACCOUNT),
    ("event_tracking", "bronze", "events/event_tracking/", ADLS_ACCOUNT),
    ("event_metadata", "bronze", "events/event_metadata/", ADLS_ACCOUNT),
]

print(f"Cache dir: {CACHE_DIR}")
for name, container, path, storage in TABLES:
    sync_table(name, container, path, storage)

## Register -- mount cached tables into DuckDB

In [ ]:
conn = duckdb.connect()

registered, missing = [], []
for name, *_ in TABLES:
    if register_local(conn, name):
        registered.append(name)
    else:
        missing.append(name)

print(f"Local cache (fast): {registered}")
if missing:
    print(f"Not yet synced -- run Sync cell above: {missing}")

## Queries

All queries below run against local Parquet files -- no cluster, no network after sync.

In [ ]:
# Row counts and status breakdown
conn.sql("""
    SELECT status, COUNT(*) AS cnt
    FROM raw_events
    GROUP BY status
    ORDER BY cnt DESC
""").df()

In [ ]:
# Join events with tracking state
conn.sql("""
    SELECT
        e.event_id,
        e.event_timestamp,
        e.status,
        t.processing_status,
        t.retry_count
    FROM raw_events e
    LEFT JOIN event_tracking t ON e.event_id = t.event_id
    WHERE t.processing_status = 'pending'
    ORDER BY e.event_timestamp DESC
    LIMIT 50
""").df()

In [ ]:
# Cache age -- when was each table last synced?
import json
from datetime import datetime, timezone

for name, *_ in TABLES:
    manifest_path = CACHE_DIR / name / ".manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        age_h = (datetime.now(timezone.utc).timestamp() - manifest_path.stat().st_mtime) / 3600
        print(f"{name}: v{manifest['version']}  {len(manifest['files'])} files  {age_h:.1f}h old")
    else:
        print(f"{name}: not cached")